# Pre-processing MultiplEYE Data

This notebook provides a step-by-step guide through how to process the eye-tracking data and the psychometric tests data collected within the MultiplEYE project. This goal of this notebook is twofold:

1. To provide a step-by-step guide on how to preprocess MultiplEYE data using the `pymovements` library and our custom preprocessing functions.
2. To serve as a tutorial for researchers who want to preprocess their own MultiplEYE data, or data from other eye-tracking datasets, using the `pymovements` library.

## Preparation steps
1. Download the data folder from the online repository. Note that this is only possible if you have access to at least one data collection protected folder. You will have access if you are an active member of one data collection group. Download the entire content of the folder.
When you download it from SwitchDrive, it will automatically create a .tar file.
2. Add the folder to the `data/` folder in this repo. The name of the folder is the data collection name, e.g., `MultiplEYE_ZH_CH_Zurich_1_2025`.
3. Extract the .tar file in the `data/` folder.
4. Make sure that the folder structure is correct. It should look like the one online and like this (there might be more data but this is not relevant at this point):
```
	MultiplEYE_ZH_CH_Zurich_1_2025/
		documentation/
		eye-tracking-sessions/
			001_.../
			002_.../
			...
			pilot_sessions/
				001_.../
				002_.../
				...
		psychometric-tests-sessions/
		stimuli_MultiplEYE_ZH_CH_Zurich_1_2025/
		...
```

## The config file



The pipeline uses a config file which can be used to specify parameters and settings for the preprocessing. It is typically named `multipleye_settings_preprocessing.yaml`. You can load it explicitly or rely on the default loading mechanism (CWD, environment variable, or legacy root).

Once you have your config file ready, you can load it as shown below.

In [41]:
# from preprocessing.data_collection.multipleye_data_collection import prepare_language_folder
from preprocessing.data_collection.multipleye_data_collection import (
    MultipleyeDataCollection,
)

import preprocessing

# the settings will be loaded into general config module, so we can access all settings at the same place
from preprocessing import settings

from preprocessing.scripts.prepare_language_folder import prepare_language_folder

import polars as pl

from preprocessing.metrics.reading.words import (
    mark_skipped_tokens,
    all_tokens_from_aois,
)

from pymovements.measure.reading.processing import compute_reading_measures

In [2]:
# If you have a specific config file, load it here:
settings.load_from_yaml("multipleye_settings_preprocessing.yaml")

In [3]:
# get the data collection name from the settings and create the path to the data folder
print(f"Active Data Collection: {settings.DATA_COLLECTION_NAME}")
print(f"Dataset Directory: {settings.DATASET_DIR}")

Active Data Collection: MultiplEYE_SV_CH_Zurich_1_2026
Dataset Directory: /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/data/MultiplEYE_SV_CH_Zurich_1_2026


### Inspecting and overriding configuration

After loading the config, you can inspect which sessions are included or excluded, and override these values for the current session without modifying the YAML file.

In [5]:
print(f"Include pilots:  {settings.INCLUDE_PILOTS}")
print(f"Included:        {settings.INCLUDE_SESSIONS}")
print(f"Excluded:        {settings.EXCLUDE_SESSIONS}")
print(f"Output dir:      {settings.OUTPUT_DIR}")
print(f"Run preflight:   {settings.RUN_PREFLIGHT_CHECK}")
print(f"Overwrite:       {settings.OVERWRITE}")

# Override example (uncomment to limit processing to specific sessions):
# settings.INCLUDE_SESSIONS = ["014_DE_DE_1_ET1", "023_DE_DE_1_ET1"]
# settings.EXCLUDE_SESSIONS = []

Include pilots:  True
Included:        ['003_SV_CH_1_ET1']
Excluded:        []
Output dir:      /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026
Run preflight:   True
Overwrite:       True


## MultiplEYE-specific preprocessing & cleaning

In order to be able to run a more generic preprocessing, the MultiplEYE data folder for one language needs to be cleaned and organized in a specific way. Running the script below will:
- unzip session folders if needed
- move session folders from core_sessions folder to the top folder
- check if there is a config file in the stimuli folder (if not, the stimulus folder was probably not uploaded correctly)
- check if there are psychometric tests (if applicable)
	- if necessary, restructure the psychometric test folder.

These steps are very individual for this data collection and results from bugs or changes across the years of collecting data.

Note that executing the cell below for the first time can take very long. However, it will run through quickly after this initial run.

In [6]:
# run the preparation function to prepare the language folder structure
prepare_language_folder()

2026-08-03 08:49:36,034 - preprocessing - INFO - Copying stimulus assets to /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026...


Next, we create a `MultipleyeDataCollection` object from the data folder. This will allow us to easily access the sessions and their information in the next steps.

In [7]:
multipleye = MultipleyeDataCollection.create_from_data_folder(
    settings.DATASET_DIR,
    include_pilots=settings.INCLUDE_PILOTS,
    excluded_sessions=settings.EXCLUDE_SESSIONS,
    included_sessions=settings.INCLUDE_SESSIONS,
)

2026-08-03 08:49:38,157 - preprocessing - INFO - Lab config loaded from /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/config/config_sv_ch_Zurich_1_2026.py
2026-08-03 08:49:38,159 - preprocessing - INFO - JSON lab config loaded from /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/config/MultiplEYE_SV_CH_Zurich_1_2026_lab_configuration.json
2026-08-03 08:49:38,163 - preprocessing - INFO - MultipleyeDataCollection initialized. data_root: /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/data/MultiplEYE_SV_CH_Zurich_1_2026/eye-tracking-sessions
2026-08-03 08:49:38,166 - preprocessing - INFO - Main config loaded from /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/config/config_sv_ch_Zurich_1.py
2026

### Preflight check

Before processing, run a preflight check to validate the dataset structure and catch common issues (missing files, incorrect folder layout, etc.). In case EDF files are missing, you can use `settings.EXCLUDE_SESSIONS = []` to exclude specific sessions, as shown a few cells above.

In [8]:
preprocessing.run_preflight_check(multipleye)

2026-08-03 08:49:41,155 - preprocessing - INFO - 
  Preflight check — all input files found


## Stage 0: Converting EDF to ASC and Preparing Session-Level Information

Stage 0 refers to the initial steps of preprocessing, which involve converting raw eye-tracking data from its original format (e.g., EDF) into a more accessible format (e.g., ASC), and preparing session-level information. This stage is specific to EyeLink eye-trackers and can be omitted for other eye-trackers.

In [9]:
multipleye.convert_edf_to_asc()

2026-08-03 08:49:43,563 - preprocessing - INFO - Starting EDF to ASC conversion for 1 sessions.
Converting EDF to ASC: 100%|██████████| 1/1 [00:00<00:00, 1033.59it/s]
2026-08-03 08:49:43,573 - preprocessing - INFO - EDF to ASC conversion completed.


Once this conversion has been completed, we can load all sessions and parse the .asc files.

In [10]:
multipleye.prepare_session_level_information()

Preparing session 003_SV_CH_1_ET1: 100%|██████████| 1/1 [00:11<00:00, 11.46s/it]


In [11]:
# print an overview on the data collection and the sessions
multipleye

Title	MultiplEYE_SV_CH_Zurich_1_2026
Dataset_type	MultiplEYE
Number_of_sessions	0
Number_of_pilots	1
Tested_language	SV
Country	CH
Year	2026
Number of eye-tracking (ET) sessions per participant	1

## Stage 1: Extracting Gaze Samples

In the first preprocessing stage, we extract gaze samples from the .asc files and create a gaze dataframe for each session. This dataframe contains the raw gaze data, including the x and y coordinates of the gaze, the timestamp. We also save the raw gaze data in a separate file for each session.

The next steps are performed for one session only. It is always possible to loop over all sessions and apply the same preprocessing steps to each of them, but for the sake of clarity and simplicity, we will work with one session as an example.



In [12]:
# pick only one session as an example to work with in the next steps
sessions = list(multipleye)  # list of Session objects
sess = sessions[0]  # a real Session
sid = sess.sid  # get the session ID (Sid) from the Session
sid

Sid(pid='003', lang='SV', country='CH', lab='1', session='ET1', session_id=1, postfix='')

In [13]:
type(sess)

preprocessing.data_collection.session.Session

### Creating Gaze Frame from ASCII File

In [14]:
gaze = preprocessing.load_gaze_data(
    asc_file=sess.asc_path,
    lab_config=sess.lab_config,
    sid=sess.sid,
    trial_cols=settings.TRIAL_COLS,
    messages=settings.ANSWER_MSG_PATTERNS,  # for comprehension questions later - see Stage 4.
)

In [15]:
# save gaze and metadata
preprocessing.save_raw_data(sid, gaze)
preprocessing.save_session_metadata(sid, gaze)

In [16]:
sid.raw_data_dir, sid.metadata_dir

(PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/raw_data/003_SV_CH_1_ET1'),
 PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/metadata/003_SV_CH_1_ET1'))

### Output directory structure

All preprocessed data is organised by data type under `preprocessed_data/<data_collection_name>/`. Each data type folder contains one subfolder per session:

```
preprocessed_data/<dcn>/
├── raw_data/
│   └── <session_save_name>/
├── fixations/
│   └── <session_save_name>/
├── saccades/
│   └── <session_save_name>/
├── scanpaths/
│   └── <session_save_name>/
├── reading_measures/
│   └── <session_save_name>/
├── sanity_checks/
│   └── <session_save_name>/
├── metadata/
│   └── <session_save_name>/
│       ├── gaze_metadata.json
│       ├── experiment.yaml
│       ├── calibrations.tsv
│       ├── calibrations.feather
│       ├── validations.tsv
│       ├── validations.feather
│       └── <session_idf>_overview.yaml
├── participant_data.csv
├── <dcn>_overview.yaml
└── stimuli_<dcn>/
```

The `Sid` object provides convenient properties to access each path:

In [17]:
print(f"Raw data:        {sid.raw_data_dir}")
print(f"Metadata:        {sid.metadata_dir}")
print(f"Fixations:       {sid.fixations_dir}")
print(f"Saccades:        {sid.saccades_dir}")
print(f"Scanpaths:       {sid.scanpaths_dir}")
print(f"Reading measures: {sid.reading_measures_dir}")

Raw data:        /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/raw_data/003_SV_CH_1_ET1
Metadata:        /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/metadata/003_SV_CH_1_ET1
Fixations:       /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/fixations/003_SV_CH_1_ET1
Saccades:        /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/saccades/003_SV_CH_1_ET1
Scanpaths:       /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/scanpaths/003_SV_CH_1_ET1
Reading measures: /home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/reading_measures/003_SV_CH_1_ET1


In order to have the metadata which is extracted by pymovements available to create out session overview, we get this information from pymovements and store it in our session object.

In [18]:
sess.pm_gaze_metadata = gaze._metadata
sess.calibrations = gaze.calibrations
sess.validations = gaze.validations

### Coordinate and Velocity Preprocessing

Eye movements are recorded in screen pixel coordinates, which depend on stimulus size and monitor setup. To compare gaze behavior across participants, screens, or datasets, it is standard to convert pixel positions 
into **degrees of visual angle (dva)**. Next, we compute **gaze velocity**, which allows us to detect saccades and distinguish them from fixations.

In [19]:
# inspect the gaze samples
gaze.samples.head()

time,pupil,page,stimulus,trial,practice,activity,session,pixel
i64,f64,str,str,str,bool,str,str,list[f64]
2787713,799.0,"""page_1""","""Enc_WikiMoon_13""","""PRACTICE_trial_1""",true,"""reading""","""003_SV_CH_1_ET1""","[78.9, 93.2]"
2787714,802.0,"""page_1""","""Enc_WikiMoon_13""","""PRACTICE_trial_1""",true,"""reading""","""003_SV_CH_1_ET1""","[78.7, 93.3]"
2787715,804.0,"""page_1""","""Enc_WikiMoon_13""","""PRACTICE_trial_1""",true,"""reading""","""003_SV_CH_1_ET1""","[79.2, 93.3]"
2787716,801.0,"""page_1""","""Enc_WikiMoon_13""","""PRACTICE_trial_1""",true,"""reading""","""003_SV_CH_1_ET1""","[79.9, 91.8]"
2787717,807.0,"""page_1""","""Enc_WikiMoon_13""","""PRACTICE_trial_1""",true,"""reading""","""003_SV_CH_1_ET1""","[79.5, 92.3]"


In [20]:
preprocessing.preprocess_gaze(gaze)

In [21]:
# inspect the preprocessed gaze samples, the dataframe should now also contain a position in dva and velocity columns
gaze.samples.head()

time,pupil,page,stimulus,trial,practice,activity,session,pixel,position,velocity
i64,f64,str,str,str,bool,str,str,list[f64],list[f64],list[f64]
2787713,799.0,"""page_1""","""Enc_WikiMoon_13""","""PRACTICE_trial_1""",true,"""reading""","""003_SV_CH_1_ET1""","[78.9, 93.2]","[-15.151475, -10.741041]","[-0.962259, -3.001021]"
2787714,802.0,"""page_1""","""Enc_WikiMoon_13""","""PRACTICE_trial_1""",true,"""reading""","""003_SV_CH_1_ET1""","[78.7, 93.3]","[-15.156524, -10.738466]","[-0.925034, -3.194814]"
2787715,804.0,"""page_1""","""Enc_WikiMoon_13""","""PRACTICE_trial_1""",true,"""reading""","""003_SV_CH_1_ET1""","[79.2, 93.3]","[-15.143902, -10.738466]","[-0.807383, -3.420028]"
2787716,801.0,"""page_1""","""Enc_WikiMoon_13""","""PRACTICE_trial_1""",true,"""reading""","""003_SV_CH_1_ET1""","[79.9, 91.8]","[-15.126228, -10.777098]","[-0.721037, -3.634057]"
2787717,807.0,"""page_1""","""Enc_WikiMoon_13""","""PRACTICE_trial_1""",true,"""reading""","""003_SV_CH_1_ET1""","[79.5, 92.3]","[-15.136328, -10.764222]","[-0.624866, -3.77264]"


## Stage 2a: Detect Events and Compute Their Properties

Eye-tracking data are typically segmented into events, i.e. `fixations` and `saccades`. Fixations represent moments when the eyes remain relatively still, allowing visual information to be processed, while saccades are the rapid movements between fixations that reposition the gaze. Detecting these events and computing their properties, such as `dispersion`, fixation `duration`, saccade `amplitude`, and `peak velocity`, provides the foundation for analyzing visual behavior and understanding how participants explore a stimulus.

### Fixations

We can detect fixations by applying the `I-VT` or the `I-DT` method.

The **I-VT (Velocity-Threshold Identification)** method distinguishes fixation and saccade points based on their point-to-point velocities. Each point is classified as a fixation if its velocity is below the specified threshold. Consecutive fixation points are then merged into a single fixation. A threshold of 20 degrees/second is commonly used as a default maximum value. Read more about [the IVT algorithm in the documentation](https://pymovements.readthedocs.io/en/stable/reference/api/pymovements.events.detection.ivt.html) 

The **I-DT (Dispersion-Threshold Identification)** method finds fixations by grouping consecutive points within a maximum separation (dispersion) threshold and a minimum duration threshold. The algorithm slides a moving window across the data: if the dispersion within the window is below the threshold, the window represents a fixation and is gradually expanded until the dispersion exceeds the threshold.
Read more about [our implementation of the IDT method](https://pymovements.readthedocs.io/en/stable/reference/api/pymovements.events.detection.idt.html).

We use the `I-VT` algorithm with the following key deafault parameters:
- `minimum duration`: 100 ms 
- `velocity threshold`: 20.0

Such properties as `location`, containing the centroid coordinates of each fixation, and `dispersion` will also be calculated.

In [22]:
preprocessing.detect_fixations(
    gaze,
)

### Saccades

Saccades are rapid eye movements that shift the point of fixation from one location to another. We detect saccades (or micro-saccades) from the velocity sequence of gaze data using the [microsaccades algorithm](https://pymovements.readthedocs.io/en/stable/reference/api/pymovements.events.detection.microsaccades.html#pymovements.events.detection.microsaccades). This algorithm implements a noise-adaptive velocity threshold, meaning that the detection threshold automatically scales with the noise level of the velocity signal. Such properties as `amplitude` and `peak velocity` of the detected saccades will also be calcuated.

The key default parameters are:
- `threshold_factor`: Multiplier used to determine the velocity threshold relative to the noise level of the signal. The default value is 6. A higher factor makes the algorithm more conservative (detects fewer saccades), while a lower factor makes it more sensitive.
- `minimum_duration`: Defines how long a velocity peak must persist to be classified as a saccade. The duration is expressed in the same units as timesteps. If no timesteps are provided, the value refers to the number of samples (default = 6), which corresponds to about 12 ms at a 500 Hz sampling rate. Shorter events are ignored as noise. 

In [23]:
preprocessing.detect_saccades(
    gaze,
)

Save our events data.

In [24]:
preprocessing.save_events_data(
    settings.FIXATION,
    sess.sid,
    split_column="trial",
    name_columns=["trial", "stimulus"],
    file_columns=["onset", "duration", "location_x", "location_y", "page"],
    data=gaze,
)

preprocessing.save_events_data(
    settings.SACCADE,
    sess.sid,
    split_column="trial",
    name_columns=["trial", "stimulus"],
    file_columns=[
        "onset",
        "duration",
        "amplitude",
        "peak_velocity",
        "dispersion",
        "page",
    ],
    data=gaze,
)

In [25]:
str(sess.sid), sess.sid

('003_SV_CH_1_ET1',
 Sid(pid='003', lang='SV', country='CH', lab='1', session='ET1', session_id=1, postfix=''))

In [26]:
sess.stimuli

[Stimulus(id=1, name='PopSci_MultiplEYE', type='experiment', pages=[StimulusPage(number=1, text='MultiplEYE-projektet\n\nNamnet ”MultiplEYE” är en ordlek som kombinerar ”multilingualism” eller ”multiple languages” med ”eye” från ”eye-tracking”. MultiplEYE är en COST-aktion finansierad av den Europeiska unionen. COST-aktioner är forskningsnätverk som stöds av European Cooperation in Science and Technology, förkortat COST. Som finansiär stöder COST vårt växande nätverk av forskare i och utanför Europa genom att ge ekonomiskt stöd till genomförandet av olika networkingevent.', image_path=PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/stimuli_images_sv_ch_1/popsci_multipleye_id1_page_1_sv.png'), aoi_image_path=PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/aoi_stimuli_images_sv

## Stage 2b: Map Fixations to AOIs

Once we have the fixations, we can map each of them to the AOIs of the stimulus. The resulting scanpath can then be saved. Note that this features is not yet completely finished.

In [27]:
preprocessing.map_fixations_to_aois(gaze, sess.stimuli)

In [28]:
gaze.events.frame

trial,stimulus,page,name,onset,offset,duration,dispersion,amplitude,peak_velocity,dispersion_right,location_x,location_y,char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,word_idx,word_idx_in_line,word
str,str,str,str,i64,i64,i64,f64,f64,f64,f64,f64,f64,i64,str,f64,f64,i64,f64,i64,i64,i64,i64,str
"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2787801,2788220,419,0.646787,null,null,null,80.56,116.225238,null,null,null,null,null,null,null,null,null,null,null
"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2788249,2788555,306,0.465144,null,null,null,115.054072,109.445603,2,"""n""",109.0,89.0,14,31.0,2,0,0,0,"""Månen"""
"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2788611,2788798,187,0.395564,null,null,null,270.165957,197.248936,18,"""k""",263.0,149.45,14,89.9,13,1,1,0,"""https://en.wikipedia.org/wiki/…"
"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2788858,2789065,207,0.581458,null,null,null,124.820192,100.329327,3,"""e""",123.0,89.0,14,31.0,3,0,0,0,"""Månen"""
"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2789108,2789247,139,0.27659,null,null,null,141.168571,177.386429,9,"""s""",137.0,149.45,14,89.9,4,1,1,0,"""https://en.wikipedia.org/wiki/…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""trial_10""","""PopSci_MultiplEYE_1""","""question_1132""","""saccade""",7947151,7947216,65,null,6.594378,176.767101,7.364846,null,null,null,null,null,null,null,null,null,null,null,null,null
"""trial_10""","""PopSci_MultiplEYE_1""","""question_1132""","""saccade""",7947445,7947505,60,null,4.424131,120.834528,4.955198,null,null,null,null,null,null,null,null,null,null,null,null,null
"""trial_10""","""PopSci_MultiplEYE_1""","""question_1132""","""saccade""",7947637,7947698,61,null,4.568003,123.125023,5.186925,null,null,null,null,null,null,null,null,null,null,null,null,null


### New Method

Taken from the annotate fixations method in fixations.py

In [29]:

group_columns = [settings.TRIAL_COL, settings.STIMULUS_COL, settings.PAGE_COL]

only_fix = (
        gaze.events.frame.filter(
            (pl.col("name") == settings.FIXATION)
            & (pl.col(settings.WORD_IDX_COL).is_not_null())
        )
        .with_row_count("fixation_id")
        .sort(group_columns + ["onset"])
    )

2026-08-03 09:16:13,441 - py.warnings - PYWARN:WARNING - /tmp/ipykernel_8363/944958957.py:8: DeprecationWarning: `DataFrame.with_row_count` is deprecated; use `with_row_index` instead. Note that the default column name has changed from 'row_nr' to 'index'.
  .with_row_count("fixation_id")



In [30]:
only_fix

fixation_id,trial,stimulus,page,name,onset,offset,duration,dispersion,amplitude,peak_velocity,dispersion_right,location_x,location_y,char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,word_idx,word_idx_in_line,word
u32,str,str,str,str,i64,i64,i64,f64,f64,f64,f64,f64,f64,i64,str,f64,f64,i64,f64,i64,i64,i64,i64,str
0,"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2788249,2788555,306,0.465144,null,null,null,115.054072,109.445603,2,"""n""",109.0,89.0,14,31.0,2,0,0,0,"""Månen"""
1,"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2788611,2788798,187,0.395564,null,null,null,270.165957,197.248936,18,"""k""",263.0,149.45,14,89.9,13,1,1,0,"""https://en.wikipedia.org/wiki/…"
2,"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2788858,2789065,207,0.581458,null,null,null,124.820192,100.329327,3,"""e""",123.0,89.0,14,31.0,3,0,0,0,"""Månen"""
3,"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2789108,2789247,139,0.27659,null,null,null,141.168571,177.386429,9,"""s""",137.0,149.45,14,89.9,4,1,1,0,"""https://en.wikipedia.org/wiki/…"
4,"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2789292,2789500,208,0.240297,null,null,null,245.860287,180.65933,16,"""w""",235.0,149.45,14,89.9,11,1,1,0,"""https://en.wikipedia.org/wiki/…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
5930,"""trial_9""","""Lit_MagicMountain_6""","""page_6""","""fixation""",7414521,7414621,100,0.248947,null,null,null,810.30198,398.558416,292,""" """,809.0,329.25,14,89.9,52,3,57,10,"""Guds"""
5931,"""trial_9""","""Lit_MagicMountain_6""","""page_6""","""fixation""",7414667,7414874,207,0.354613,null,null,null,913.411058,400.014423,299,"""a""",907.0,329.25,14,89.9,59,3,58,11,"""namn"""
5932,"""trial_9""","""Lit_MagicMountain_6""","""page_6""","""fixation""",7414918,7415138,220,0.339244,null,null,null,996.307692,397.732579,305,"""d""",991.0,329.25,14,89.9,65,3,59,12,"""ändå"""


In [31]:
stim = sess.stimuli[2]
aois = stim.text_stimulus.aois
words_only = all_tokens_from_aois(aois, trial=stim.trial_id)
words_only = words_only.with_columns(pl.lit(stim.name).alias("stimulus"))
words_only = words_only.filter(pl.col("page")=="page_2")
words_only

trial,page,word_idx,word,stimulus
str,str,i64,str,str
"""trial_5""","""page_2""",0,"""Utbildning,""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",1,"""studier""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",2,"""och""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",3,"""praktik""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",4,"""i""","""Ins_LearningMobility"""
…,…,…,…,…
"""trial_5""","""page_2""",58,"""och""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",59,"""stöd""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",60,"""via""","""Ins_LearningMobility"""


In [32]:
only_fix = only_fix.filter((pl.col("trial")=="trial_5") & (pl.col("page")=="page_2"))
only_fix

fixation_id,trial,stimulus,page,name,onset,offset,duration,dispersion,amplitude,peak_velocity,dispersion_right,location_x,location_y,char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,word_idx,word_idx_in_line,word
u32,str,str,str,str,i64,i64,i64,f64,f64,f64,f64,f64,f64,i64,str,f64,f64,i64,f64,i64,i64,i64,i64,str
3161,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4868248,4868377,129,0.310876,null,null,null,146.190769,111.883846,4,"""l""",137.0,89.0,14,31.0,4,0,0,0,"""Utbildning,"""
3162,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4868429,4868586,157,0.201681,null,null,null,299.068354,115.499367,15,"""d""",291.0,89.0,14,31.0,15,0,1,1,"""studier"""
3163,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4868873,4869060,187,0.388902,null,null,null,314.170213,112.65266,16,"""i""",305.0,89.0,14,31.0,16,0,1,1,"""studier"""
3164,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4869219,4869391,172,0.265423,null,null,null,461.349711,101.728324,27,"""k""",459.0,89.0,14,31.0,27,0,3,3,"""praktik"""
3165,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4869439,4869578,139,0.276457,null,null,null,590.053571,103.326429,36,"""t""",585.0,89.0,14,31.0,36,0,5,5,"""ett"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3220,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4882743,4882977,234,0.400617,null,null,null,260.703404,643.74766,447,"""r""",249.0,598.95,14,89.9,12,6,62,1,"""Erasmus+."""
3221,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4883051,4883211,160,0.494209,null,null,null,674.52236,547.249689,399,""" """,669.0,509.05,14,89.9,42,5,55,8,"""ett"""
3222,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4883257,4883417,160,0.396542,null,null,null,778.280124,538.448447,406,"""r""",767.0,509.05,14,89.9,49,5,56,9,"""särskilt"""


In [33]:
rm_df = compute_reading_measures(
    fixations=only_fix,
    aois = words_only,
    word_index_column= "word_idx",
    word_column = "word"
)

In [102]:
rm_df

word,word_index,FFD,SFD,FD,FPRT,FRT,TFT,RRT,RPD_inc,RPD_exc,RBRT,Fix,FPF,RR,FPReg,TRC_out,TRC_in,SL_in,SL_out,TFC
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""studier""",0,157,0,157,344,344,344,0,344,0,344,1,1,0,0,0,0,1,2,2
"""och""",1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""praktik""",2,172,172,172,172,172,172,0,172,0,172,1,1,0,0,0,0,2,2,1
"""i""",3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""ett""",4,139,139,139,139,139,139,0,139,0,139,1,1,0,0,0,0,2,3,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""och""",57,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""stöd""",58,141,141,141,141,141,246,105,141,0,141,1,1,1,0,1,0,2,2,2
"""via""",59,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Old Method

In [37]:
# The resulting mapping can be stored as a scanpath, which is a sequence of AOIs that were fixated in the order they were fixated.
preprocessing.save_scanpaths(sid, gaze)

In [38]:
# save metadata again
preprocessing.save_session_metadata(sid, gaze)

In [39]:
old_rm_df = preprocessing.calculate_reading_measures(gaze, sess.stimuli)
preprocessing.save_reading_measures(sid, old_rm_df)

In [42]:
new_rm_df = preprocessing.new_calculate_reading_measures(gaze, sess.stimuli)
preprocessing.save_reading_measures(sid, new_rm_df)

AttributeError: module 'preprocessing' has no attribute 'new_calculate_reading_measures'

In [40]:
old_rm_df.filter((pl.col("trial") == "trial_10") & (pl.col("page")=="page_1"))

trial,page,word_idx,word,stimulus,skipped,TFC,FD,FFD,FPRT,FRT,RRT,FPFC,TRC_in,TRC_out,LP,SL_in,SL_out,RPD_inc,RPD_exc,RBRT,TFT,FPF,RR,SFD
str,str,i64,str,str,i8,u32,i64,i64,i64,i64,i64,u32,u32,u32,i64,i64,i64,i64,i64,i64,i64,i8,i8,i64
"""trial_10""","""page_1""",0,"""MultiplEYE-projektet""","""PopSci_MultiplEYE""",0,2,157,157,337,337,0,2,0,0,5,0,1,337,0,337,337,1,0,0
"""trial_10""","""page_1""",1,"""Namnet""","""PopSci_MultiplEYE""",0,1,155,155,155,155,0,1,0,0,22,1,1,155,0,155,155,1,0,155
"""trial_10""","""page_1""",2,"""”MultiplEYE”""","""PopSci_MultiplEYE""",0,7,193,193,314,314,2205,2,2,0,31,1,3,314,0,314,2519,1,1,0
"""trial_10""","""page_1""",3,"""är""","""PopSci_MultiplEYE""",1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""trial_10""","""page_1""",4,"""en""","""PopSci_MultiplEYE""",0,1,140,0,0,140,140,0,0,0,45,2,1,0,0,0,140,0,1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""trial_10""","""page_1""",57,"""till""","""PopSci_MultiplEYE""",1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""trial_10""","""page_1""",58,"""genomförandet""","""PopSci_MultiplEYE""",0,1,142,142,142,142,0,1,0,0,434,3,2,142,0,142,142,1,0,142
"""trial_10""","""page_1""",59,"""av""","""PopSci_MultiplEYE""",1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## Pymovements weirdness

In [106]:
fixations = only_fix
aois = words_only
word_index_column= "word_idx"
word_column = "word"

In [107]:
fixations

fixation_id,trial,stimulus,page,name,onset,offset,duration,dispersion,location_x,location_y,char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,word_idx,word_idx_in_line,word,amplitude,peak_velocity,dispersion_right
u32,str,str,str,str,i64,i64,i64,f64,f64,f64,i64,str,f64,f64,i64,f64,i64,i64,i64,i64,str,f64,f64,f64
3161,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4868248,4868377,129,0.310876,146.190769,111.883846,4,"""l""",137.0,89.0,14,31.0,4,0,0,0,"""Utbildning,""",null,null,null
3162,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4868429,4868586,157,0.201681,299.068354,115.499367,15,"""d""",291.0,89.0,14,31.0,15,0,1,1,"""studier""",null,null,null
3163,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4868873,4869060,187,0.388902,314.170213,112.65266,16,"""i""",305.0,89.0,14,31.0,16,0,1,1,"""studier""",null,null,null
3164,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4869219,4869391,172,0.265423,461.349711,101.728324,27,"""k""",459.0,89.0,14,31.0,27,0,3,3,"""praktik""",null,null,null
3165,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4869439,4869578,139,0.276457,590.053571,103.326429,36,"""t""",585.0,89.0,14,31.0,36,0,5,5,"""ett""",null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3220,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4882743,4882977,234,0.400617,260.703404,643.74766,447,"""r""",249.0,598.95,14,89.9,12,6,62,1,"""Erasmus+.""",null,null,null
3221,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4883051,4883211,160,0.494209,674.52236,547.249689,399,""" """,669.0,509.05,14,89.9,42,5,55,8,"""ett""",null,null,null
3222,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4883257,4883417,160,0.396542,778.280124,538.448447,406,"""r""",767.0,509.05,14,89.9,49,5,56,9,"""särskilt""",null,null,null


In [108]:
dummy_fixation_dict: dict[str, list[int] | list[str]] = {}
for col, dtype in fixations.schema.items():
    if dtype == pl.String:
        dummy_fixation_dict[col] = ['']
    else:
        dummy_fixation_dict[col] = [0]
dummy_fixation = pl.DataFrame(
    dummy_fixation_dict,
    schema=fixations.schema,
)
fixations = pl.concat([fixations, dummy_fixation])

In [109]:
fixations

fixation_id,trial,stimulus,page,name,onset,offset,duration,dispersion,location_x,location_y,char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,word_idx,word_idx_in_line,word,amplitude,peak_velocity,dispersion_right
u32,str,str,str,str,i64,i64,i64,f64,f64,f64,i64,str,f64,f64,i64,f64,i64,i64,i64,i64,str,f64,f64,f64
3161,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4868248,4868377,129,0.310876,146.190769,111.883846,4,"""l""",137.0,89.0,14,31.0,4,0,0,0,"""Utbildning,""",null,null,null
3162,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4868429,4868586,157,0.201681,299.068354,115.499367,15,"""d""",291.0,89.0,14,31.0,15,0,1,1,"""studier""",null,null,null
3163,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4868873,4869060,187,0.388902,314.170213,112.65266,16,"""i""",305.0,89.0,14,31.0,16,0,1,1,"""studier""",null,null,null
3164,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4869219,4869391,172,0.265423,461.349711,101.728324,27,"""k""",459.0,89.0,14,31.0,27,0,3,3,"""praktik""",null,null,null
3165,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4869439,4869578,139,0.276457,590.053571,103.326429,36,"""t""",585.0,89.0,14,31.0,36,0,5,5,"""ett""",null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3221,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4883051,4883211,160,0.494209,674.52236,547.249689,399,""" """,669.0,509.05,14,89.9,42,5,55,8,"""ett""",null,null,null
3222,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4883257,4883417,160,0.396542,778.280124,538.448447,406,"""r""",767.0,509.05,14,89.9,49,5,56,9,"""särskilt""",null,null,null
3223,"""trial_5""","""Ins_LearningMobility_3""","""page_2""","""fixation""",4883470,4883678,208,0.317105,936.883732,550.678469,418,"""ä""",935.0,509.05,14,89.9,61,5,57,10,"""riktmärke""",null,null,null


In [111]:
words_only

trial,page,word_idx,word,stimulus
str,str,i64,str,str
"""trial_5""","""page_2""",0,"""Utbildning,""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",1,"""studier""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",2,"""och""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",3,"""praktik""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",4,"""i""","""Ins_LearningMobility"""
…,…,…,…,…
"""trial_5""","""page_2""",58,"""och""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",59,"""stöd""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",60,"""via""","""Ins_LearningMobility"""


In [110]:
aois = aois.with_columns(
        (pl.col(word_index_column) - 1).alias(word_index_column),
    )
aois

trial,page,word_idx,word,stimulus
str,str,i64,str,str
"""trial_5""","""page_2""",-1,"""Utbildning,""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",0,"""studier""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",1,"""och""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",2,"""praktik""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",3,"""i""","""Ins_LearningMobility"""
…,…,…,…,…
"""trial_5""","""page_2""",57,"""och""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",58,"""stöd""","""Ins_LearningMobility"""
"""trial_5""","""page_2""",59,"""via""","""Ins_LearningMobility"""


In [114]:
word_indices = aois[word_index_column].to_list()
words = aois[word_column].to_list()
word_indices

[-1,
 0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61]

In [115]:
rm_dict = {
        word_index: {
            'word': word,
            'word_index': word_index,
            'FFD': 0, 'SFD': 0, 'FD': 0, 'FPRT': 0, 'FRT': 0, 'TFT': 0, 'RRT': 0,
            'RPD_inc': 0, 'RPD_exc': 0, 'RBRT': 0, 'Fix': 0, 'FPF': 0, 'RR': 0,
            'FPReg': 0, 'TRC_out': 0, 'TRC_in': 0, 'SL_in': 0, 'SL_out': 0, 'TFC': 0,
        } for word_index, word in zip(word_indices, words)
    }
rm_dict

{-1: {'word': 'Utbildning,',
  'word_index': -1,
  'FFD': 0,
  'SFD': 0,
  'FD': 0,
  'FPRT': 0,
  'FRT': 0,
  'TFT': 0,
  'RRT': 0,
  'RPD_inc': 0,
  'RPD_exc': 0,
  'RBRT': 0,
  'Fix': 0,
  'FPF': 0,
  'RR': 0,
  'FPReg': 0,
  'TRC_out': 0,
  'TRC_in': 0,
  'SL_in': 0,
  'SL_out': 0,
  'TFC': 0},
 0: {'word': 'studier',
  'word_index': 0,
  'FFD': 0,
  'SFD': 0,
  'FD': 0,
  'FPRT': 0,
  'FRT': 0,
  'TFT': 0,
  'RRT': 0,
  'RPD_inc': 0,
  'RPD_exc': 0,
  'RBRT': 0,
  'Fix': 0,
  'FPF': 0,
  'RR': 0,
  'FPReg': 0,
  'TRC_out': 0,
  'TRC_in': 0,
  'SL_in': 0,
  'SL_out': 0,
  'TFC': 0},
 1: {'word': 'och',
  'word_index': 1,
  'FFD': 0,
  'SFD': 0,
  'FD': 0,
  'FPRT': 0,
  'FRT': 0,
  'TFT': 0,
  'RRT': 0,
  'RPD_inc': 0,
  'RPD_exc': 0,
  'RBRT': 0,
  'Fix': 0,
  'FPF': 0,
  'RR': 0,
  'FPReg': 0,
  'TRC_out': 0,
  'TRC_in': 0,
  'SL_in': 0,
  'SL_out': 0,
  'TFC': 0},
 2: {'word': 'praktik',
  'word_index': 2,
  'FFD': 0,
  'SFD': 0,
  'FD': 0,
  'FPRT': 0,
  'FRT': 0,
  'TFT': 0,
 

--> Problem: Now a catch-all entry at index -1 would be added, but this removes the first word of the page

## Iteration through pages

In [123]:
#Creating dataframe with just 
stim = sess.stimuli[2]
aois = stim.text_stimulus.aois
words_only = all_tokens_from_aois(aois, trial=stim.trial_id)
words_only = words_only.with_columns(pl.lit(stim.name).alias("stimulus"))

#Only taking fixations on words into account
group_columns = [settings.TRIAL_COL, settings.STIMULUS_COL, settings.PAGE_COL]

only_fix = (
        gaze.events.frame.filter(
            (pl.col("name") == settings.FIXATION)
            & (pl.col(settings.WORD_IDX_COL).is_not_null())
        )
        .with_row_count("fixation_id")
        .sort(group_columns + ["onset"])
    )
only_fix = only_fix.filter(pl.col("trial")=="trial_5")

2026-07-29 14:38:58,774 - py.warnings - PYWARN:WARNING - /tmp/ipykernel_16150/3523474209.py:15: DeprecationWarning: `DataFrame.with_row_count` is deprecated; use `with_row_index` instead. Note that the default column name has changed from 'row_nr' to 'index'.
  .with_row_count("fixation_id")



In [128]:
for page in stim.pages:
    page_idx = f"page_{page.number}"
    page_words = words_only.filter(pl.col("page")==page_idx)
    page_fix = only_fix.filter(pl.col("page")==page_idx)
    rm_df = compute_reading_measures(
        fixations=page_fix,
        aois = page_words,
        word_index_column= "word_idx",
        word_column = "word"
    )
    print(rm_df.head())

    

shape: (5, 21)
┌──────────────┬────────────┬─────┬─────┬───┬────────┬───────┬────────┬─────┐
│ word         ┆ word_index ┆ FFD ┆ SFD ┆ … ┆ TRC_in ┆ SL_in ┆ SL_out ┆ TFC │
│ ---          ┆ ---        ┆ --- ┆ --- ┆   ┆ ---    ┆ ---   ┆ ---    ┆ --- │
│ str          ┆ i64        ┆ i64 ┆ i64 ┆   ┆ i64    ┆ i64   ┆ i64    ┆ i64 │
╞══════════════╪════════════╪═════╪═════╪═══╪════════╪═══════╪════════╪═════╡
│ FRÅN         ┆ 0          ┆ 113 ┆ 113 ┆ … ┆ 2      ┆ 1     ┆ 1      ┆ 3   │
│ KOMMISSIONEN ┆ 1          ┆ 132 ┆ 132 ┆ … ┆ 0      ┆ 1     ┆ -1     ┆ 3   │
│ TILL         ┆ 2          ┆ 461 ┆ 461 ┆ … ┆ 0      ┆ 2     ┆ 2      ┆ 1   │
│ RÅDET:       ┆ 3          ┆ 0   ┆ 0   ┆ … ┆ 0      ┆ 2     ┆ 3      ┆ 1   │
│ Lägesrapport ┆ 4          ┆ 114 ┆ 0   ┆ … ┆ 1      ┆ 2     ┆ 2      ┆ 5   │
└──────────────┴────────────┴─────┴─────┴───┴────────┴───────┴────────┴─────┘
shape: (5, 21)
┌─────────┬────────────┬─────┬─────┬───┬────────┬───────┬────────┬─────┐
│ word    ┆ word_index ┆ FFD ┆ SFD ┆ … 

## Iteration through stimuli and pages

In [34]:
settings.WORD_IDX_COL

'word_idx'

In [35]:
group_columns = [settings.TRIAL_COL, settings.STIMULUS_COL, settings.PAGE_COL]

only_fix = (
        gaze.events.frame.filter(
            (pl.col("name") == settings.FIXATION)
            & (pl.col(settings.WORD_IDX_COL).is_not_null())
        )
        .with_row_count("fixation_id")
        .sort(group_columns + ["onset"])
    )
only_fix

2026-08-03 09:17:16,121 - py.warnings - PYWARN:WARNING - /tmp/ipykernel_8363/4184453923.py:8: DeprecationWarning: `DataFrame.with_row_count` is deprecated; use `with_row_index` instead. Note that the default column name has changed from 'row_nr' to 'index'.
  .with_row_count("fixation_id")



fixation_id,trial,stimulus,page,name,onset,offset,duration,dispersion,amplitude,peak_velocity,dispersion_right,location_x,location_y,char_idx,char,top_left_x,top_left_y,width,height,char_idx_in_line,line_idx,word_idx,word_idx_in_line,word
u32,str,str,str,str,i64,i64,i64,f64,f64,f64,f64,f64,f64,i64,str,f64,f64,i64,f64,i64,i64,i64,i64,str
0,"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2788249,2788555,306,0.465144,null,null,null,115.054072,109.445603,2,"""n""",109.0,89.0,14,31.0,2,0,0,0,"""Månen"""
1,"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2788611,2788798,187,0.395564,null,null,null,270.165957,197.248936,18,"""k""",263.0,149.45,14,89.9,13,1,1,0,"""https://en.wikipedia.org/wiki/…"
2,"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2788858,2789065,207,0.581458,null,null,null,124.820192,100.329327,3,"""e""",123.0,89.0,14,31.0,3,0,0,0,"""Månen"""
3,"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2789108,2789247,139,0.27659,null,null,null,141.168571,177.386429,9,"""s""",137.0,149.45,14,89.9,4,1,1,0,"""https://en.wikipedia.org/wiki/…"
4,"""PRACTICE_trial_1""","""Enc_WikiMoon_13""","""page_1""","""fixation""",2789292,2789500,208,0.240297,null,null,null,245.860287,180.65933,16,"""w""",235.0,149.45,14,89.9,11,1,1,0,"""https://en.wikipedia.org/wiki/…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
5930,"""trial_9""","""Lit_MagicMountain_6""","""page_6""","""fixation""",7414521,7414621,100,0.248947,null,null,null,810.30198,398.558416,292,""" """,809.0,329.25,14,89.9,52,3,57,10,"""Guds"""
5931,"""trial_9""","""Lit_MagicMountain_6""","""page_6""","""fixation""",7414667,7414874,207,0.354613,null,null,null,913.411058,400.014423,299,"""a""",907.0,329.25,14,89.9,59,3,58,11,"""namn"""
5932,"""trial_9""","""Lit_MagicMountain_6""","""page_6""","""fixation""",7414918,7415138,220,0.339244,null,null,null,996.307692,397.732579,305,"""d""",991.0,329.25,14,89.9,65,3,59,12,"""ändå"""


In [36]:
rm_all_trials = []

for stim in sess.stimuli:
    aois = stim.text_stimulus.aois
    words_only = all_tokens_from_aois(aois, trial=stim.trial_id)
    words_only = words_only.with_columns(pl.lit(stim.name).alias("stimulus"))
    trial_idx = stim.trial_id

    for page in stim.pages:
        page_idx = f"page_{page.number}"
        page_words = words_only.filter(pl.col("page")==page_idx)
        page_fix = only_fix.filter((pl.col("trial")==trial_idx) & (pl.col("page")==page_idx))
        rm = compute_reading_measures(
            fixations=page_fix,
            aois = page_words,
            word_index_column= "word_idx",
            word_column = "word"
        )
        rm = rm.with_columns(pl.lit(trial_idx).alias("trial"), pl.lit(page_idx).alias("page"), pl.lit(stim.name).alias("stimulus"))
        rm_all_trials.append(rm)

rm_df = pl.concat(rm_all_trials)
rm_df

word,word_index,FFD,SFD,FD,FPRT,FRT,TFT,RRT,RPD_inc,RPD_exc,RBRT,Fix,FPF,RR,FPReg,TRC_out,TRC_in,SL_in,SL_out,TFC,trial,page,stimulus
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str
"""Namnet""",0,155,155,155,155,155,155,0,155,0,155,1,1,0,0,0,0,1,1,1,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
"""”MultiplEYE”""",1,193,0,193,314,314,2519,2205,314,0,314,1,1,1,0,0,2,1,3,7,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
"""är""",2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
"""en""",3,0,0,140,0,140,140,140,0,0,0,1,0,1,0,0,0,2,1,1,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
"""ordlek""",4,162,162,162,162,162,475,313,162,0,162,1,1,1,0,0,1,3,1,3,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""plats""",29,213,213,213,213,213,213,0,213,0,213,1,1,0,0,0,0,1,3,1,"""PRACTICE_trial_2""","""page_2""","""Lit_NorthWind"""
"""för""",30,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""PRACTICE_trial_2""","""page_2""","""Lit_NorthWind"""
"""att""",31,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""PRACTICE_trial_2""","""page_2""","""Lit_NorthWind"""


In [148]:
sess.stimuli[0]

Stimulus(id=1, name='PopSci_MultiplEYE', type='experiment', pages=[StimulusPage(number=1, text='MultiplEYE-projektet\n\nNamnet ”MultiplEYE” är en ordlek som kombinerar ”multilingualism” eller ”multiple languages” med ”eye” från ”eye-tracking”. MultiplEYE är en COST-aktion finansierad av den Europeiska unionen. COST-aktioner är forskningsnätverk som stöds av European Cooperation in Science and Technology, förkortat COST. Som finansiär stöder COST vårt växande nätverk av forskare i och utanför Europa genom att ge ekonomiskt stöd till genomförandet av olika networkingevent.', image_path=PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/stimuli_images_sv_ch_1/popsci_multipleye_id1_page_1_sv.png'), aoi_image_path=PosixPath('/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/preprocessed_data/MultiplEYE_SV_CH_Zurich_1_2026/stimuli_MultiplEYE_SV_CH_Zurich_1_2026/aoi_stimuli_images_sv_

## ADD SETTINGS PARAMETERS

In [153]:
group_columns = [settings.TRIAL_COL, settings.STIMULUS_COL, settings.PAGE_COL]

fixation_table = (
        gaze.events.frame.filter(
            (pl.col("name") == settings.FIXATION)
            & (pl.col(settings.WORD_IDX_COL).is_not_null())
        )
        .with_row_count("fixation_id")
        .sort(group_columns + ["onset"])
    )


rm_all_trials = []

for stim in sess.stimuli:
    aois = stim.text_stimulus.aois
    words_only = all_tokens_from_aois(aois, trial=stim.trial_id)
    words_only = words_only.with_columns(pl.lit(stim.name).alias(settings.STIMULUS_COL))
    trial_idx = stim.trial_id

    for page in stim.pages:
        page_idx = f"page_{page.number}"
        page_words = words_only.filter(pl.col(settings.PAGE_COL)==page_idx)
        page_fix = only_fix.filter((pl.col(settings.TRIAL_COL)==trial_idx) & (pl.col(settings.PAGE_COL)==page_idx))
        rm = compute_reading_measures(
            fixations=page_fix,
            aois = page_words,
            word_index_column= settings.WORD_IDX_COL,
            word_column = "word"
        )
        rm = rm.with_columns(pl.lit(trial_idx).alias(settings.TRIAL_COL), pl.lit(page_idx).alias(settings.PAGE_COL), pl.lit(stim.name).alias(settings.STIMULUS_COL))
        rm_all_trials.append(rm)

rm_df = pl.concat(rm_all_trials)
rm_df = rm_df.rename({"word_index":settings.WORD_IDX_COL})
rm_df

2026-07-29 16:04:04,368 - py.warnings - PYWARN:WARNING - /tmp/ipykernel_16150/4074227793.py:8: DeprecationWarning: `DataFrame.with_row_count` is deprecated; use `with_row_index` instead. Note that the default column name has changed from 'row_nr' to 'index'.
  .with_row_count("fixation_id")



word,word_idx,FFD,SFD,FD,FPRT,FRT,TFT,RRT,RPD_inc,RPD_exc,RBRT,Fix,FPF,RR,FPReg,TRC_out,TRC_in,SL_in,SL_out,TFC,trial,page,stimulus
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str
"""Namnet""",0,155,155,155,155,155,155,0,155,0,155,1,1,0,0,0,0,1,1,1,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
"""”MultiplEYE”""",1,193,0,193,314,314,2519,2205,314,0,314,1,1,1,0,0,2,1,3,7,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
"""är""",2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
"""en""",3,0,0,140,0,140,140,140,0,0,0,1,0,1,0,0,0,2,1,1,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
"""ordlek""",4,162,162,162,162,162,475,313,162,0,162,1,1,1,0,0,1,3,1,3,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""plats""",29,213,213,213,213,213,213,0,213,0,213,1,1,0,0,0,0,1,3,1,"""PRACTICE_trial_2""","""page_2""","""Lit_NorthWind"""
"""för""",30,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""PRACTICE_trial_2""","""page_2""","""Lit_NorthWind"""
"""att""",31,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""PRACTICE_trial_2""","""page_2""","""Lit_NorthWind"""


In [151]:
settings.WORD_IDX_COL

'word_idx'

In [152]:
rm_df.rename({"word_index":settings.WORD_IDX_COL})

word,word_idx,FFD,SFD,FD,FPRT,FRT,TFT,RRT,RPD_inc,RPD_exc,RBRT,Fix,FPF,RR,FPReg,TRC_out,TRC_in,SL_in,SL_out,TFC,trial,page,stimulus
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str
"""Namnet""",0,155,155,155,155,155,155,0,155,0,155,1,1,0,0,0,0,1,1,1,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
"""”MultiplEYE”""",1,193,0,193,314,314,2519,2205,314,0,314,1,1,1,0,0,2,1,3,7,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
"""är""",2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
"""en""",3,0,0,140,0,140,140,140,0,0,0,1,0,1,0,0,0,2,1,1,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
"""ordlek""",4,162,162,162,162,162,475,313,162,0,162,1,1,1,0,0,1,3,1,3,"""trial_10""","""page_1""","""PopSci_MultiplEYE"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""plats""",29,213,213,213,213,213,213,0,213,0,213,1,1,0,0,0,0,1,3,1,"""PRACTICE_trial_2""","""page_2""","""Lit_NorthWind"""
"""för""",30,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""PRACTICE_trial_2""","""page_2""","""Lit_NorthWind"""
"""att""",31,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""PRACTICE_trial_2""","""page_2""","""Lit_NorthWind"""
